In [1]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-5" 

In [3]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
        "stop_sequences" : stop_sequences
    }

    if system:
        params["system"] = system
        
    response = client.messages.create(**params)
    # response.content can include a ThinkingBlock before the TextBlock
    # (adaptive thinking is on by default on this model), so find the
    # first text block instead of assuming content[0] is text.
    return next(block.text for block in response.content if block.type == "text")   


In [4]:
# Make a start listing of messages
messages = []


# Add in the initial user message
add_user_message(messages, "Generate a very short event bridge rule as JSON")
add_assistant_message(messages,"```json")

# Pass the message to chat() to get a response from the model
text = chat(messages, stop_sequences=["```"])


print(text)




{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}



In [5]:
import json
json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [ ]:
messages = []
prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)
add_assistant_message(messages,"Here are three commands in a single block without any comments:\n```bash")

# "\n" alone isn't a valid stop sequence (the API requires each stop
# sequence to contain non-whitespace), so stop on the closing code fence.
text = chat(messages, stop_sequences=["```"])
text.strip()
text


In [ ]:
%pip install ipython

In [10]:
from IPython.display import Markdown
Markdown(text.strip())

Here are three short AWS CLI commands:

```bash
aws s3 ls

aws ec2 describe-instances

aws iam list-users
```